# 02. Reproduce LightGBM and tune PPG-only alternatives

This notebook is a reproducible research driver around the repository scripts.
It uses only relative repository paths and writes generated data/results under
ignored `data/processed/` and `artifacts/` directories.

Primary public benchmark:
- target: self-reported discrete emotion (`EmotType`, exposed as `reported`)
- split: participant-grouped cross-validation
- metrics: top-1/top-2/top-3, macro F1, weighted F1, balanced accuracy, Cohen's kappa, confusion matrix

In [ ]:
from pathlib import Path
import subprocess

ROOT = Path.cwd()
DATA_ROOT = ROOT / "data" / "extracted"
FEATURE_ALL = ROOT / "data" / "processed" / "features_all.parquet"
FEATURE_PPG = ROOT / "data" / "processed" / "features_ppg.parquet"
FEATURE_PPG_RICH = ROOT / "data" / "processed" / "features_ppg_rich.parquet"
RESULTS = ROOT / "artifacts" / "results"
OPTUNA_BASE = ROOT / "artifacts" / "optuna" / "ppg_reported_grouped"
OPTUNA_RICH = ROOT / "artifacts" / "optuna" / "ppg_rich_reported_grouped"
WORKERS = 4
OPTUNA_TRIALS = 20

def run(cmd: list[str], *, skip_if_exists: Path | None = None) -> None:
    if skip_if_exists is not None and skip_if_exists.exists():
        print(f"[skip] {skip_if_exists}")
        return
    print("$", " ".join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], check=True)

print(ROOT)

## 1. Prepare LLaMAC v6 data

This downloads Figshare v6 files, verifies checksums, extracts participant archives,
and writes `data/processed/dataset_index.csv`. Existing valid files are reused.

In [ ]:
run(["uv", "run", "python", "scripts/download_llamac.py", "--prepare"], skip_if_exists=ROOT / "data" / "processed" / "dataset_index.csv")

## 2. Build feature tables

- `all`: official-notebook-style all-channel features (EEG, GSR, PPG, SKT, RESP)
- `ppg`: official-notebook-style PPG-only features
- `ppg_rich`: PPG-only extension with derivative, spectral, peak, and segment features

In [ ]:
run(["uv", "run", "--group", "ml", "python", "scripts/build_features.py", "--mode", "all", "--workers", str(WORKERS), "--output", FEATURE_ALL], skip_if_exists=FEATURE_ALL)
run(["uv", "run", "--group", "ml", "python", "scripts/build_features.py", "--mode", "ppg", "--workers", str(WORKERS), "--output", FEATURE_PPG], skip_if_exists=FEATURE_PPG)
run(["uv", "run", "--group", "ml", "python", "scripts/build_features.py", "--mode", "ppg_rich", "--workers", str(WORKERS), "--output", FEATURE_PPG_RICH], skip_if_exists=FEATURE_PPG_RICH)

## 3. Reproduce LightGBM baselines

Grouped results are the leakage-safe public benchmark. Official-style stratified
results are included only to compare with the source notebook protocol.

In [ ]:
run(["uv", "run", "--group", "ml", "python", "scripts/train_model.py", "--features", FEATURE_ALL, "--model", "lightgbm", "--feature-set", "all", "--target", "reported", "--split", "grouped", "--folds", "5", "--device", "cpu", "--output", RESULTS / "lightgbm_all_reported_grouped.json"], skip_if_exists=RESULTS / "lightgbm_all_reported_grouped.json")
run(["uv", "run", "--group", "ml", "python", "scripts/train_model.py", "--features", FEATURE_ALL, "--model", "lightgbm", "--feature-set", "all", "--target", "reported", "--split", "stratified", "--official-exclusions", "--folds", "5", "--device", "cpu", "--output", RESULTS / "lightgbm_all_reported_official_stratified.json"], skip_if_exists=RESULTS / "lightgbm_all_reported_official_stratified.json")
run(["uv", "run", "--group", "ml", "python", "scripts/train_model.py", "--features", FEATURE_ALL, "--model", "lightgbm", "--feature-set", "all", "--target", "intended", "--split", "stratified", "--official-exclusions", "--folds", "5", "--device", "cpu", "--output", RESULTS / "lightgbm_all_intended_official_stratified.json"], skip_if_exists=RESULTS / "lightgbm_all_intended_official_stratified.json")
run(["uv", "run", "--group", "ml", "python", "scripts/train_model.py", "--features", FEATURE_PPG, "--model", "lightgbm", "--feature-set", "ppg", "--target", "reported", "--split", "grouped", "--folds", "5", "--device", "auto", "--output", RESULTS / "lightgbm_ppg_reported_grouped.json"], skip_if_exists=RESULTS / "lightgbm_ppg_reported_grouped.json")
run(["uv", "run", "--group", "ml", "python", "scripts/train_model.py", "--features", FEATURE_PPG, "--model", "lightgbm", "--feature-set", "ppg", "--target", "reported", "--split", "stratified", "--official-exclusions", "--folds", "5", "--device", "cpu", "--output", RESULTS / "lightgbm_ppg_reported_official_stratified.json"], skip_if_exists=RESULTS / "lightgbm_ppg_reported_official_stratified.json")
run(["uv", "run", "--group", "ml", "python", "scripts/train_model.py", "--features", FEATURE_PPG, "--model", "lightgbm", "--feature-set", "ppg", "--target", "intended", "--split", "stratified", "--official-exclusions", "--folds", "5", "--device", "cpu", "--output", RESULTS / "lightgbm_ppg_intended_official_stratified.json"], skip_if_exists=RESULTS / "lightgbm_ppg_intended_official_stratified.json")

## 4. Tune PPG-only alternatives with Optuna

The base PPG study tunes several classical/tree families. The rich PPG study tunes
the two families that improved the PPG-only macro-F1 ceiling in the current run.

In [ ]:
run(["uv", "run", "--group", "ml", "python", "scripts/tune_models.py", "--features", FEATURE_PPG, "--feature-set", "ppg", "--target", "reported", "--split", "grouped", "--folds", "5", "--models", "lightgbm", "extra_trees", "hist_gradient_boosting", "random_forest", "logistic_regression", "--trials", str(OPTUNA_TRIALS), "--metric", "macro_f1", "--device", "cpu", "--output-dir", OPTUNA_BASE], skip_if_exists=OPTUNA_BASE / "locked-results" / "logistic_regression_ppg_reported_grouped_best.json")
run(["uv", "run", "--group", "ml", "python", "scripts/tune_models.py", "--features", FEATURE_PPG, "--feature-set", "ppg", "--target", "reported", "--split", "grouped", "--folds", "5", "--models", "svc_rbf", "--trials", "10", "--metric", "macro_f1", "--device", "cpu", "--output-dir", OPTUNA_BASE], skip_if_exists=OPTUNA_BASE / "locked-results" / "svc_rbf_ppg_reported_grouped_best.json")
run(["uv", "run", "--group", "ml", "python", "scripts/tune_models.py", "--features", FEATURE_PPG_RICH, "--feature-set", "ppg", "--target", "reported", "--split", "grouped", "--folds", "5", "--models", "lightgbm", "logistic_regression", "--trials", str(OPTUNA_TRIALS), "--metric", "macro_f1", "--device", "cpu", "--output-dir", OPTUNA_RICH], skip_if_exists=OPTUNA_RICH / "locked-results" / "logistic_regression_ppg_reported_grouped_best.json")

## 5. Summarize comparable results

In [ ]:
summary_path = RESULTS / "summary_with_rich.csv"
run(["uv", "run", "python", "scripts/summarize_results.py", RESULTS, OPTUNA_BASE / "locked-results", OPTUNA_RICH / "locked-results", "--output", summary_path])
print(summary_path.read_text()[:4000])